### Load golden answers

In [21]:
import json

with open('data/ptbr-multiple-choice-qa-pairs.json', 'r', encoding='utf-8') as f:
    golden_qa_pairs = json.load(f)

### Load model answers

In [22]:
responses_file = 'qwen3-235b-a22b-ptbr-responses-prompt-language-ptbr.json'

responses = []
with open(f'results/{responses_file}', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():  # skip empty lines
            responses.append(json.loads(line))


### Parse model final answers

In [23]:
import re

nr_responses = 0
unanswered = 0

for response in responses:
    try:
        text = response["raw_response"]["choice.message.content"]
    except KeyError:
        # skip responses without the content field
        response['final_answer'] = None
        continue

    m = re.search(r'(?<=\\boxed\{)([A-Za-z])(?=\}(?!.*\\boxed))', text)
    if m:
        response["final_answer"] = m.group(0)   # store the letter string, not the match object
        nr_responses += 1
    else:
        unanswered += 1
        response["final_answer"] = None
        print("No match for id", response.get('id'))
        print("Response text:", response["raw_response"]["choice.message.content"][-30:])
        print("Finish Reason:", response["raw_response"].get("finish_reason"))
        print("\n\n\n")
        
print(f"Number of responses processed: {nr_responses}")
print(f"Number of unanswered responses: {unanswered}")


No match for id 1182
Response text: sas é:

$$
\boxed{\text{A}}
$$
Finish Reason: stop




No match for id 1224
Response text: tos é:

$$
\boxed{\text{C}}
$$
Finish Reason: stop




No match for id 1250
Response text: ove amigos é:

$$
\boxed{4}
$$
Finish Reason: stop




No match for id 1257
Response text: la):**

$$
\boxed{\text{E}}
$$
Finish Reason: stop




No match for id 1302
Response text: ed{120} $ km.

**Resposta: C**
Finish Reason: stop




No match for id 1318
Response text: eta é:

$$
\boxed{\text{A}}
$$
Finish Reason: stop




No match for id 1399
Response text: orreta

$$
\boxed{\text{D}}
$$
Finish Reason: stop




No match for id 1437
Response text: ria**:

$$
\boxed{\text{C}}
$$
Finish Reason: stop




No match for id 1496
Response text: final:

$$
\boxed{\text{A}}
$$
Finish Reason: stop




No match for id 1413
Response text: resents is:**

$$
\boxed{7}
$$
Finish Reason: stop




Number of responses processed: 518
Number of unanswered responses: 10


### Calculate accuracies

In [24]:
import re
from collections import defaultdict

# build lookup tables
responses_by_id = {r.get('id'): r for r in responses}
golden_by_id = {g['id']: g for g in golden_qa_pairs}

# initialize counters per level (levels 1..4)
levels = [1, 2, 3, 4]
stats = {
    lvl: {
        'total_gold': 0,
        'answered': 0,
        'correct_answered': 0,
        'correct_including_missing': 0
    }
    for lvl in levels
}

# also track counters split by whether the question contains a LaTeX figure
figure_cats = [True, False]
stats_by_figure = {
    fig: {
        lvl: {
            'total_gold': 0,
            'answered': 0,
            'correct_answered': 0,
            'correct_including_missing': 0
        }
        for lvl in levels
    }
    for fig in figure_cats
}

# iterate gold entries (ensures we only evaluate ids that have gold answers)
for gid, golden in golden_by_id.items():
    try:
        lvl = int(golden.get('level', 0))
    except (TypeError, ValueError):
        continue
    if lvl not in levels:
        continue

    fig = golden.get('contains_latex_figure_in_question', False)
    if isinstance(fig, str):
        fig = fig.strip().lower() in ('true', '1', 'yes', 'y')
    fig = bool(fig)

    stats[lvl]['total_gold'] += 1
    stats_by_figure[fig][lvl]['total_gold'] += 1

    resp = responses_by_id.get(gid)
    final_ans = None

    if resp is not None:
        fa = resp.get('final_answer', None)
        if fa is not None:
            if hasattr(fa, 'group'):
                try:
                    final_ans = fa.group(0)
                except Exception:
                    final_ans = None
            else:
                final_ans = fa

    correct_option = str(golden.get('correct_option', '')).strip().lower()

    if final_ans is not None:
        stats[lvl]['answered'] += 1
        stats_by_figure[fig][lvl]['answered'] += 1
        final_norm = str(final_ans).strip().lower()
        if final_norm == correct_option:
            stats[lvl]['correct_answered'] += 1
            stats[lvl]['correct_including_missing'] += 1
            stats_by_figure[fig][lvl]['correct_answered'] += 1
            stats_by_figure[fig][lvl]['correct_including_missing'] += 1

# compute global aggregates
global_totals = {
    'total_gold': sum(stats[l]['total_gold'] for l in levels),
    'answered': sum(stats[l]['answered'] for l in levels),
    'correct_answered': sum(stats[l]['correct_answered'] for l in levels),
    'correct_including_missing': sum(stats[l]['correct_including_missing'] for l in levels)
}

# compute global aggregates split by figure presence
global_totals_by_figure = {
    fig: {
        'total_gold': sum(stats_by_figure[fig][l]['total_gold'] for l in levels),
        'answered': sum(stats_by_figure[fig][l]['answered'] for l in levels),
        'correct_answered': sum(stats_by_figure[fig][l]['correct_answered'] for l in levels),
        'correct_including_missing': sum(stats_by_figure[fig][l]['correct_including_missing'] for l in levels)
    }
    for fig in figure_cats
}

def percent(num, denom):
    return f"{(num/denom*100):.2f}%" if denom else "N/A"

# --- WRITE TO FILE INSTEAD OF PRINT ---
output_path = "results/" + responses_file + "-acc-report.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write("Per-level results (levels 1..4):\n\n")
    for l in levels:
        s = stats[l]
        f.write(f"Level {l}:\n")
        f.write(f"  Golden items (denominator for 'including missing') : {s['total_gold']}\n")
        f.write(f"  Answered (have final_answer)                          : {s['answered']}\n")
        f.write(f"  Correct among answered                               : {s['correct_answered']}\n")
        f.write(f"  Accuracy (answered only)                             : {percent(s['correct_answered'], s['answered'])}\n")
        f.write(f"  Accuracy (missing counted as incorrect)              : {percent(s['correct_including_missing'], s['total_gold'])}\n\n")

    f.write("GLOBAL (all levels combined):\n")
    f.write(f"  Golden items total: {global_totals['total_gold']}\n")
    f.write(f"  Answered total     : {global_totals['answered']}\n")
    f.write(f"  Correct (answered) : {global_totals['correct_answered']}\n")
    f.write(f"  Accuracy (answered only)         : {percent(global_totals['correct_answered'], global_totals['answered'])}\n")
    f.write(f"  Accuracy (missing counted wrong) : {percent(global_totals['correct_including_missing'], global_totals['total_gold'])}\n")

    f.write("\nBy question figure presence:\n")
    for fig in figure_cats:
        label = "WITH figures" if fig else "WITHOUT figures"
        f.write(f"\n{label}:\n")
        for l in levels:
            s = stats_by_figure[fig][l]
            f.write(f"Level {l}:\n")
            f.write(f"  Golden items (denominator for 'including missing') : {s['total_gold']}\n")
            f.write(f"  Answered (have final_answer)                          : {s['answered']}\n")
            f.write(f"  Correct among answered                               : {s['correct_answered']}\n")
            f.write(f"  Accuracy (answered only)                             : {percent(s['correct_answered'], s['answered'])}\n")
            f.write(f"  Accuracy (missing counted as incorrect)              : {percent(s['correct_including_missing'], s['total_gold'])}\n\n")

        g = global_totals_by_figure[fig]
        f.write(f"GLOBAL ({label}):\n")
        f.write(f"  Golden items total: {g['total_gold']}\n")
        f.write(f"  Answered total     : {g['answered']}\n")
        f.write(f"  Correct (answered) : {g['correct_answered']}\n")
        f.write(f"  Accuracy (answered only)         : {percent(g['correct_answered'], g['answered'])}\n")
        f.write(f"  Accuracy (missing counted wrong) : {percent(g['correct_including_missing'], g['total_gold'])}\n")

print(f"Accuracy report written to: {output_path}")


Accuracy report written to: results/qwen3-235b-a22b-ptbr-responses-prompt-language-ptbr.json-acc-report.txt
